# Cycle topology - RQ2 campaign

Ring network (0-1-2-3-0): every agent has exactly **2** neighbours, so it is
degree-regular like the clique but sparse like the line. That separates
*symmetry* from *connectivity*, which the star cannot.

**Before running:** Accelerator = `GPU T4 x2`, Internet = `On`, and a Kaggle
Secret named `HF_TOKEN` for gated models (Gemma, Llama).

This notebook is deliberately a thin launcher: all the campaign logic lives in
`campaign.py` in the repo, so pulling the repo picks up any fix. Kaggle
executes the cells saved in *your* workspace and never the notebook stored in
the repo, so logic kept in cells can only be fixed by hand-editing cells.

Edit `MODEL` and `SESSION` in the last cell. Session A is
baseline+no_sense+silence+counterfactual, session B the three framings; every
model needs both except Qwen3-4B, where `--session ALL` fits in one sitting.
For runs over ~2 h use **Save Version -> Save & Run All** so a browser
disconnect cannot kill the session.


In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch, transformers
assert torch.cuda.is_available(), "GPU not enabled! Settings -> Accelerator -> GPU T4"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU:  ", torch.cuda.get_device_name(0), f"({gb:.1f} GB)")
print("torch:", torch.__version__, " transformers:", transformers.__version__)


In [ ]:
GITHUB_REPO = "https://github.com/stsimpe/cheaptalk_bench.git"

import os, subprocess
if os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "-C", "/kaggle/working/repo", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", GITHUB_REPO, "/kaggle/working/repo"], check=True)
%cd /kaggle/working/repo
head = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print("HEAD:", head.strip())


In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HUGGINGFACE_API_KEY"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded (needed for Gemma / Llama)")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen):", e)


## Launch

`campaign.py` resolves the scenarios and the per-model `max_new_tokens` from
the campaign plan, runs the sweep, verifies the runs actually produced (count
and recorded topology) and only then writes the zip. It exits non-zero if
anything is off, and `check=True` turns that into a failed cell rather than a
silently empty zip.

Run `!python campaign.py --help` to see every knob, or add `--dry-run` to print
the plan without spending GPU time.


In [ ]:
import subprocess, sys

MODEL   = "google/gemma-2-2b-it"
# MODEL = "Qwen/Qwen3-4B"                  # this one can take --session ALL
# MODEL = "Qwen/Qwen2.5-7B-Instruct"
# MODEL = "meta-llama/Llama-3.1-8B-Instruct"
# MODEL = "google/gemma-2-9b-it"

SESSION = "A"     # "A", "B", or "ALL"

subprocess.run([sys.executable, "campaign.py",
                "--model", MODEL,
                "--session", SESSION,
                "--topology", "cycle"], check=True)


## After the run

1. Download the zip printed above into `diplomatikh/drive_sync/` and upload it
   to the Drive folder `cheaptalk_bench_results`.
2. Append the runs to the `Runs` sheet of `cheaptalk_results_tracker.xlsx`;
   `Combinations` and `Results` recompute themselves.
3. Add a row to `TRACK_RECORD.md` with the wall-clock time printed by the run,
   so the remaining estimates can be recalibrated.
